# v11 - OOF-teacher targets on the v08 ConvNeXt-tiny trainer (GPU T4 x2)

Design: `docs/research/v11-strategy-revision-design.md` revision 2.1, decision D-014.

**Provenance.** Derived from our own `notebooks/v08-convnext-tiny.ipynb` (Kaggle
`lingxd/v08-convnext-tiny` version 2), not from public code. Changes:

- **Training target** = `0.5 * soft + 0.5 * v08 OOF`: the v06 4-source soft target mixed with
  the v08 out-of-fold prediction (K_infer 16) of the same study, computed in this notebook from
  the attached v08 OOF files. Per-cell loss weights (UNK 0.3) are unchanged. The 58 gold studies
  have no OOF, are never trained on and are used for evaluation only.
- **Two arms on fold 0** (design 4.1), one per T4: A = 10 epochs, LR 5e-5 backbone / 2e-4 head;
  B = 15 epochs, LR 1e-4 / 3e-4; both `K_train = 4`. Time limits A 6 h, B 9 h, global deadline
  11 h.
- **Evaluation at `K_infer = 16`** from epoch 4: validation macro AUC (against the soft target
  rounded at 0.5) selects the best epoch, and the same checkpoint scores a fixed subset of 500
  training studies without augmentation, so training and validation loss (against the training
  target, same weights) are compared like-for-like.
- **Gate R1** (design 4.2, rev. 2.1) computed in the notebook.
- Removed from v08: the `smoke` and `auto` modes and the Gate A `K_infer` grid (K_infer 16 was
  settled by D-009).

Modes (`MODE` in the configuration cell):

- `fold0_arms` - fold 0 with arms A (GPU 0) and B (GPU 1) in parallel, then Gate R1.
- `rerun_a` - Gate R1 rule 1: fold 0 with arm A alone (6 h limit), then Gate R1 once more
  (attempt 2; if arm A is still incomplete, the rule says stop).
- `folds` - folds in `FOLDS_QUEUE` with the arm picked by Gate R1 (`QUEUE_ARM`) through a work
  queue over both GPUs; a fold starts only if its expected duration (times 1.15) fits before
  the global deadline. Per-fold time limit A 6 h, B 9 h.

An arm that stops at its time limit or fails is **incomplete** (receipt status other than
`done`) and never enters Gate R1. The time limit stops training: an arm that has trained all its
epochs is complete even if its last evaluation (about 17 minutes at K16) ends after the limit. A
process still running one hour after its limit is treated as hung, stopped and incomplete; a
non-zero exit code also makes an arm incomplete. Every arm first runs the v08 memory probe (activation
checkpointing, then a smaller micro-batch on out-of-memory).

Inputs: outputs of `lingxd/v06-dicom-audit` and `lingxd/v07-cache-320`, the private dataset
`lingxd/rsna-knee-v08-oof`; internet on for the ImageNet weights (training only).
Outputs in `/kaggle/working/v11/`: checkpoints `v11_fold_<k>_<arm>_best.pt`, receipts, OOF and
gold predictions, `v11_train_targets.csv`, Gate R1 decision.

In [ ]:
# v11 configuration
import json
import os
import subprocess
import sys
import time
from pathlib import Path

VERSION = 'v11'
MODE = 'folds'               # 'fold0_arms' (design 4.1), 'rerun_a' (Gate R1 rule 1) or 'folds' (design 4.3)
FOLD = 0                     # fold of 'fold0_arms' and 'rerun_a'
FOLDS_QUEUE = [1, 2]         # 'folds': trained through the GPU work queue
QUEUE_ARM = 'B'              # 'folds': the arm picked by Gate R1 (v11_decision_r1_attempt<n>.json)
TRAIN_PROBE_N = 0            # training studies scored like-for-like at every evaluation; 0 disables
NOTEBOOK_T0 = time.time()
GLOBAL_DEADLINE_S = 11.0 * 3600   # stop everything before Kaggle's 12 h session limit
WORK = Path('/kaggle/working') / VERSION
WORK.mkdir(parents=True, exist_ok=True)

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
OOF_WEIGHT = 0.5   # training target = (1 - OOF_WEIGHT) * soft target + OOF_WEIGHT * v08 OOF
OOF_FILES = {k: f'v08_oof_fold{k}_k4_k16.csv' for k in range(5)}

COMMON = {
    'version': VERSION,
    'backbone': 'convnext_tiny.fb_in22k_ft_in1k',
    'pretrained': True,
    'img': 320,
    'weight_decay': 0.05,
    'warmup_epochs': 1,
    'effective_batch_studies': 8,
    'grad_checkpointing': False,
    'eval_from_epoch': 4,
    'k_infer': 16,
    'train_probe_n': TRAIN_PROBE_N,
    'eval_batch_tokens': 160,
    'num_workers': 2,
    'seed': 2026,
    'device': 'cuda',
}
# Arms (design 4.1). Expected fold time (rev. 2.1, from v08 logs): about 1,010 s per training
# epoch; evaluation at about 113 tokens/s, i.e. about 670 s for K16 validation plus about 354 s
# for the 500-study training subset.
ARMS = {
    'A': {'epochs': 10, 'lr_backbone': 5e-5, 'lr_head': 2e-4, 'k_train': 4, 'micro_batch_studies': 2,
          'gpu': 0, 'time_limit_s': 6.0 * 3600, 'expected_fold_s': (4.8 if TRAIN_PROBE_N else 4.1) * 3600},
    'B': {'epochs': 15, 'lr_backbone': 1e-4, 'lr_head': 3e-4, 'k_train': 4, 'micro_batch_studies': 2,
          'gpu': 1, 'time_limit_s': 9.0 * 3600, 'expected_fold_s': (7.6 if TRAIN_PROBE_N else 6.4) * 3600},
}
# Gate R1 (design 4.2, rev. 2.1): complete -> qualified (val >= val_min) -> pick one arm by gold
# (A if within tie of the best) -> rules 2-4 on the picked arm's gold.
GATE_R1 = {'val_min': 0.871, 'gold_go': 0.915, 'gold_band_min': 0.905, 'tie': 0.005, 'quota_margin': 0.2}
R1_ATTEMPT = 2 if MODE == 'rerun_a' else 1
print(f'{VERSION}: mode={MODE}, train probe {TRAIN_PROBE_N}, arms ' +
      ', '.join(f'{a}: {c["epochs"]} ep, LR {c["lr_backbone"]}/{c["lr_head"]}, limit {c["time_limit_s"] / 3600:.1f} h'
                for a, c in ARMS.items()), flush=True)

In [ ]:
# Locate the v06, v07 and v08-OOF inputs without walking the competition image tree.
def find_unique_file(name, root=Path('/kaggle/input')):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions', 'cache')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    if len(hits) != 1:
        raise FileNotFoundError(f'expected one {name} under {root}, found {hits}')
    return hits[0]

PATHS = {
    'targets': str(find_unique_file('v06_targets.csv')),
    'folds': str(find_unique_file('v06_folds.csv')),
    'cache_studies': str(find_unique_file('v07_cache_studies.csv')),
}
PATHS['cache_dir'] = str(Path(PATHS['cache_studies']).parent / 'cache')
assert Path(PATHS['cache_dir']).is_dir(), PATHS['cache_dir']
OOF_PATHS = {k: find_unique_file(name) for k, name in OOF_FILES.items()}
print(json.dumps({**PATHS, 'oof': {k: str(v) for k, v in OOF_PATHS.items()}}, indent=2))

## Training targets (design 4.1)

For every non-gold study: `(1 - OOF_WEIGHT) * soft + OOF_WEIGHT * OOF`, where the OOF prediction
comes from the v08 model of the study's own fold (which never saw it). Checks: each OOF file
holds exactly the studies of its v06 fold, no gold study has an OOF row, every training study
has one, all values are probabilities. The table is written to `v11_train_targets.csv` and read
by the trainer; the SHA-256 of each input file goes into `v11_targets_summary.json`.

In [ ]:
import hashlib
import numpy as np
import pandas as pd

targets = pd.read_csv(PATHS['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
folds = pd.read_csv(PATHS['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
cached = pd.read_csv(PATHS['cache_studies'], dtype={'StudyInstanceUID': str})
ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
fold_of = folds.fold.reindex(targets.index)
gold_ids = set(targets.index[targets.is_gold.astype(bool)])

oof_parts, oof_sha = [], {}
for k, path in sorted(OOF_PATHS.items()):
    df = pd.read_csv(path, dtype={'StudyInstanceUID': str})
    assert df.columns.tolist() == ['StudyInstanceUID'] + LABELS, f'{path.name}: unexpected columns'
    assert df.StudyInstanceUID.is_unique, f'{path.name}: duplicate studies'
    expected = {u for u in targets.index if fold_of[u] == k and u in ok}
    assert set(df.StudyInstanceUID) == expected, f'{path.name}: studies differ from v06 fold {k}'
    oof_sha[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    oof_parts.append(df.set_index('StudyInstanceUID'))
oof = pd.concat(oof_parts)
assert oof.index.is_unique and not (set(oof.index) & gold_ids), 'OOF rows must be unique and non-gold'
values = oof[LABELS].to_numpy(np.float64)
assert np.isfinite(values).all() and values.min() >= 0.0 and values.max() <= 1.0, 'OOF values must be probabilities'

pool = [u for u in targets.index if fold_of[u] >= 0 and u in ok]
assert set(pool) == set(oof.index), 'every non-gold training study needs exactly one OOF row'
soft = targets.loc[pool, LABELS].astype(np.float64)
assert np.isfinite(soft.to_numpy()).all(), 'missing soft target'
teacher = oof.loc[pool, LABELS].astype(np.float64)
mixed = (1.0 - OOF_WEIGHT) * soft + OOF_WEIGHT * teacher
mixed.index.name = 'StudyInstanceUID'
PATHS['train_targets'] = str(WORK / 'v11_train_targets.csv')
mixed.to_csv(PATHS['train_targets'])

stats = pd.DataFrame({'soft_mean': soft.mean(), 'oof_mean': teacher.mean(), 'target_mean': mixed.mean(),
                      'soft_std': soft.std(), 'oof_std': teacher.std(), 'target_std': mixed.std()}).round(4)
print(f'training targets: {len(mixed)} studies, gold excluded: {len(gold_ids)}, OOF weight {OOF_WEIGHT}')
print(stats.to_string())
targets_summary = {'version': VERSION, 'oof_weight': OOF_WEIGHT, 'n_train_targets': int(len(mixed)),
                   'n_gold_excluded': len(gold_ids), 'oof_sha256': oof_sha,
                   'train_targets_sha256': hashlib.sha256(Path(PATHS['train_targets']).read_bytes()).hexdigest(),
                   'per_label': stats.to_dict(orient='index')}
(WORK / 'v11_targets_summary.json').write_text(json.dumps(targets_summary, indent=2))

## Trainer script

Written to disk so that each arm runs in its own process on its own GPU. The notebook is the
source of truth; the script is regenerated on every run. Data pipeline, model, optimiser,
schedule, augmentation and memory probe are unchanged from v08.

In [ ]:
%%writefile /kaggle/working/v11_train.py
"""v11 trainer: one arm on one GPU with OOF-teacher targets. Configuration comes as a JSON argument.

Unchanged from the v08 trainer: data pipeline, model, optimiser, schedule, augmentation, memory
probe. Changed: the training target (v11_train_targets.csv), per-arm epochs and learning rates,
evaluation at K_infer 16 with a like-for-like training-subset loss, best epoch selected at K16.
"""
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
N_SLOTS = 5
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


# ----------------------------------------------------------------------------- data
def load_study(cache_dir, uid):
    with np.load(Path(cache_dir) / f'{uid}.npz') as z:
        return z['jpeg'], z['offsets'], z['slot']


def decode(jpeg, offsets, k):
    return cv2.imdecode(np.frombuffer(jpeg[offsets[k]:offsets[k + 1]].tobytes(), np.uint8),
                        cv2.IMREAD_GRAYSCALE)


def slot_centres(n, k, train, rng):
    if train:
        edges = np.linspace(0, n, k + 1)
        c = np.floor(edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])).astype(int)
    else:
        c = np.linspace(0, n - 1, k).round().astype(int)
    return np.clip(c, 0, n - 1)


def augment(tri, rng):
    # tri: (3, H, W) uint8 triplet; the same geometric transform for all three channels.
    h, w = tri.shape[1:]
    angle = rng.uniform(-10, 10)
    scale = rng.uniform(0.9, 1.1)
    tx, ty = rng.uniform(-0.05, 0.05) * w, rng.uniform(-0.05, 0.05) * h
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(tri.transpose(1, 2, 0)), m, (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    img = img.astype(np.float32) / 255.0
    gamma = rng.uniform(0.85, 1.15)
    img = np.clip(img, 0, 1) ** gamma
    img = img * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05)
    return np.clip(img, 0, 1).transpose(2, 0, 1)


class StudyDataset(Dataset):
    def __init__(self, uids, targets, weights, cache_dir, k, train, img, seed):
        self.uids, self.targets, self.weights = list(uids), targets, weights
        self.cache_dir, self.k, self.train, self.img, self.seed = cache_dir, k, train, img, seed
        self.epoch = 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        jpeg, offsets, slot = load_study(self.cache_dir, self.uids[i])
        tokens, slots, cache = [], [], {}
        for s in range(N_SLOTS):
            idx = np.flatnonzero(slot == s)
            if len(idx) == 0:
                continue
            n = len(idx)
            for c in slot_centres(n, self.k, self.train, rng):
                chans = []
                for d in (-1, 0, 1):
                    j = int(idx[min(max(c + d, 0), n - 1)])
                    if j not in cache:
                        cache[j] = decode(jpeg, offsets, j)
                    chans.append(cache[j])
                tri = np.stack(chans)
                if tri.shape[1] != self.img:
                    tri = np.stack([cv2.resize(x, (self.img, self.img), interpolation=cv2.INTER_AREA) for x in tri])
                x = augment(tri, rng) if self.train else tri.astype(np.float32) / 255.0
                tokens.append((x - MEAN) / STD)
                slots.append(s)
        return {'x': np.stack(tokens).astype(np.float32), 'slot': np.asarray(slots, np.int64),
                'y': self.targets[i], 'w': self.weights[i], 'uid': self.uids[i]}


def collate(batch):
    return batch


# ---------------------------------------------------------------------------- model
class Net(nn.Module):
    def __init__(self, backbone, pretrained, n_out=12, att_dim=256, drop=0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        d = self.backbone.num_features
        self.slot_emb = nn.Embedding(N_SLOTS, d)
        self.norm = nn.LayerNorm(d)
        self.att_v = nn.Linear(d, att_dim)
        self.att_u = nn.Linear(d, att_dim)
        self.att_w = nn.Linear(att_dim, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.head_w, std=0.01)

    def encode(self, x, chunk=None):
        if chunk is None or x.shape[0] <= chunk:
            return self.backbone(x)
        return torch.cat([self.backbone(x[i:i + chunk]) for i in range(0, x.shape[0], chunk)])

    def pool(self, h, slot, counts):
        # h: (N, D) tokens of several studies concatenated; counts: tokens per study.
        h = self.norm(h + self.slot_emb(slot))
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float()  # (N, 12)
        out, start = [], 0
        for n in counts:
            hs, as_ = h[start:start + n].float(), a[start:start + n]
            wts = torch.softmax(as_, dim=0)                                  # (n, 12)
            pooled = torch.einsum('nk,nd->kd', wts, hs)                      # (12, D)
            out.append((self.drop(pooled) * self.head_w).sum(-1) + self.head_b)
            start += n
        return torch.stack(out)                                              # (B, 12)


def batch_tensors(batch, device):
    x = torch.from_numpy(np.concatenate([b['x'] for b in batch])).to(device, non_blocking=True)
    slot = torch.from_numpy(np.concatenate([b['slot'] for b in batch])).to(device)
    counts = [len(b['slot']) for b in batch]
    return x, slot, counts


# ------------------------------------------------------------------------- metrics
def macro_auc(y_true_bin, y_score):
    per = {}
    for j, name in enumerate(LABELS):
        col = y_true_bin[:, j]
        if col.min() != col.max():
            per[name] = float(roc_auc_score(col, y_score[:, j]))
    return float(np.mean(list(per.values()))) if per else float('nan'), per


def bootstrap_ci(y_true_bin, y_score, n=200, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        s = rng.integers(0, len(y_true_bin), len(y_true_bin))
        m, _ = macro_auc(y_true_bin[s], y_score[s])
        if np.isfinite(m):
            vals.append(m)
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [None, None]


def weighted_bce(logits, y, w):
    # The training loss over a whole set: per-cell BCE with logits times the cell weight, over the weight sum.
    w = torch.from_numpy(np.asarray(w, np.float32))
    cell = F.binary_cross_entropy_with_logits(torch.from_numpy(np.asarray(logits, np.float32)),
                                              torch.from_numpy(np.asarray(y, np.float32)), reduction='none')
    return float((cell * w).sum() / w.sum())


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


@torch.no_grad()
def predict(model, ds, device, batch_tokens, amp):
    # Returns study ids in dataset order and float32 logits.
    model.eval()
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=CFG['num_workers'], collate_fn=collate)
    logits, uids = [], []
    for batch in loader:
        x, slot, counts = batch_tensors(batch, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            h = model.encode(x, chunk=batch_tokens)
            out = model.pool(h, slot, counts)
        logits.append(out.float().cpu().numpy())
        uids += [b['uid'] for b in batch]
    return uids, np.concatenate(logits)


# --------------------------------------------------------------------------- main
CFG = json.loads(sys.argv[1])


def main():
    t_start = time.time()
    arm, fold, out = CFG['arm'], CFG['fold'], Path(CFG['out_dir'])
    out.mkdir(parents=True, exist_ok=True)
    stem = f'{CFG["version"]}_fold_{fold}_{arm}'             # checkpoints: v11_fold_<k>_<arm>_best.pt
    torch.manual_seed(CFG['seed'])
    device = torch.device(CFG['device'] if torch.cuda.is_available() or CFG['device'] == 'cpu' else 'cpu')
    amp = device.type == 'cuda'

    targets = pd.read_csv(CFG['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    fit = pd.read_csv(CFG['train_targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    folds = pd.read_csv(CFG['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    cached = pd.read_csv(CFG['cache_studies'], dtype={'StudyInstanceUID': str})
    ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
    fold_of = folds.fold.reindex(targets.index)
    train_ids = [u for u in targets.index if fold_of[u] >= 0 and fold_of[u] != fold and u in ok]
    val_ids = [u for u in targets.index if fold_of[u] == fold and u in ok]
    gold_ids = [u for u in targets.index if bool(targets.loc[u, 'is_gold']) and u in ok]
    assert not set(gold_ids) & (set(train_ids) | set(val_ids)), 'gold studies are never trained or validated on'
    Y_soft = targets[LABELS].to_numpy(np.float32)
    Y_fit = fit.reindex(targets.index)[LABELS].to_numpy(np.float32)    # training target; NaN for gold
    W = targets[[f'{c}__weight' for c in LABELS]].to_numpy(np.float32)
    pos = {u: i for i, u in enumerate(targets.index)}
    pick = lambda ids, m: m[[pos[u] for u in ids]]
    assert np.isfinite(pick(train_ids + val_ids, Y_fit)).all(), 'a training or validation study has no target'
    n_probe = min(int(CFG['train_probe_n']), len(train_ids))
    probe_ids = sorted(np.random.default_rng(CFG['seed']).choice(sorted(train_ids), n_probe, replace=False).tolist()) \
        if n_probe else []
    log(f'arm {arm} fold {fold}: train {len(train_ids)}, val {len(val_ids)}, gold {len(gold_ids)}, '
        f'train probe {len(probe_ids)}, device {device}')

    k = CFG['k_train']
    train_ds = StudyDataset(train_ids, pick(train_ids, Y_fit), pick(train_ids, W), CFG['cache_dir'], k, True, CFG['img'], CFG['seed'])
    mb = CFG['micro_batch_studies']
    accum = max(1, CFG['effective_batch_studies'] // mb)
    model = Net(CFG['backbone'], CFG['pretrained']).to(device)
    if CFG['grad_checkpointing']:
        model.backbone.set_grad_checkpointing(True)
    probe = {'grad_checkpointing': bool(CFG['grad_checkpointing']), 'micro_batch_studies': mb, 'attempts': []}
    if device.type == 'cuda':
        while True:
            n_tok = mb * N_SLOTS * k
            try:
                xx = torch.randn(n_tok, 3, CFG['img'], CFG['img'], device=device)
                ss = torch.arange(n_tok, device=device) % N_SLOTS
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    lo = model.pool(model.encode(xx), ss, [N_SLOTS * k] * mb)
                lo.float().sum().backward()
                model.zero_grad(set_to_none=True)
                probe['attempts'].append({'tokens': n_tok, 'ckpt': probe['grad_checkpointing'], 'ok': True,
                                          'peak_reserved_gb': round(torch.cuda.max_memory_reserved() / 1e9, 2)})
                del xx, ss, lo
                torch.cuda.empty_cache()
                break
            except torch.OutOfMemoryError:
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                probe['attempts'].append({'tokens': n_tok, 'ckpt': probe['grad_checkpointing'], 'ok': False})
                if not probe['grad_checkpointing']:
                    model.backbone.set_grad_checkpointing(True)
                    probe['grad_checkpointing'] = True
                elif mb > 1:
                    mb = mb // 2
                    probe['micro_batch_studies'] = mb
                else:
                    raise
        accum = max(1, CFG['effective_batch_studies'] // mb)
        log(f'arm {arm}: memory probe {probe}')
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
                             {'params': head_params, 'lr': CFG['lr_head']}], weight_decay=CFG['weight_decay'])
    base_lrs = [CFG['lr_backbone'], CFG['lr_head']]
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    steps_per_epoch = math.ceil(len(train_ids) / (mb * accum))
    total_steps = steps_per_epoch * CFG['epochs']
    warm = steps_per_epoch * CFG['warmup_epochs']

    def set_lr(step):
        f = step / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
        for g, lr in zip(opt.param_groups, base_lrs):
            g['lr'] = lr * f

    receipt = {'version': CFG['version'], 'arm': arm, 'fold': fold, 'status': 'running',
               'epochs': CFG['epochs'], 'lr_backbone': CFG['lr_backbone'], 'lr_head': CFG['lr_head'],
               'k_train': k, 'k_infer': CFG['k_infer'], 'time_limit_s': CFG['time_limit_s'],
               'memory_probe': probe, 'n_train': len(train_ids), 'n_val': len(val_ids), 'n_gold': len(gold_ids),
               'n_train_probe': len(probe_ids),
               'train_probe_sha1': hashlib.sha1('\n'.join(probe_ids).encode()).hexdigest()[:12],
               'train_targets_sha256': hashlib.sha256(Path(CFG['train_targets']).read_bytes()).hexdigest(),
               'micro_batch_studies': mb, 'accum': accum, 'tokens_per_study_train': N_SLOTS * k,
               'epoch_eval': {}}
    write = lambda: (out / f'{CFG["version"]}_receipt_fold{fold}_{arm}.json').write_text(json.dumps(receipt, indent=2))

    def time_up():
        return time.time() - t_start > CFG['time_limit_s']

    def save_state(tag, epoch, step):
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'step': step, 'cfg': CFG}, out / f'{stem}_{tag}.pt')

    def run_eval(ids, y):
        # Same checkpoint, no augmentation, K_infer evenly spaced centres per slot.
        ds = StudyDataset(ids, pick(ids, y), pick(ids, W), CFG['cache_dir'], CFG['k_infer'], False, CFG['img'], CFG['seed'])
        t = time.time()
        uids, z = predict(model, ds, device, CFG['eval_batch_tokens'], amp)
        assert uids == list(ids)
        return z, time.time() - t

    def eval_record(ids, z, seconds):
        # AUC against the soft target rounded at 0.5; loss against the training target and the soft target.
        m, per = macro_auc((pick(ids, Y_soft) >= 0.5).astype(np.float32), sigmoid(z))
        return {'auc': m, 'loss': weighted_bce(z, pick(ids, Y_fit), pick(ids, W)),
                'loss_soft': weighted_bce(z, pick(ids, Y_soft), pick(ids, W)), 'seconds': round(seconds, 1)}, per

    step, micro, best, best_epoch, best_per, stopped = 0, 0, -1.0, -1, {}, False
    t_data = t_compute = 0.0
    n_images = 0
    torch.cuda.reset_peak_memory_stats() if amp else None
    epoch_seconds = []
    for epoch in range(CFG['epochs']):
        train_ds.epoch = epoch
        loader = DataLoader(train_ds, batch_size=mb, shuffle=True, num_workers=CFG['num_workers'],
                            collate_fn=collate, drop_last=True, persistent_workers=False,
                            generator=torch.Generator().manual_seed(CFG['seed'] + epoch))
        model.train()
        opt.zero_grad(set_to_none=True)
        loss_sum, loss_n = 0.0, 0
        t0 = time.time()
        for batch in loader:
            t1 = time.time()
            t_data += t1 - t0
            x, slot, counts = batch_tensors(batch, device)
            y = torch.from_numpy(np.stack([b['y'] for b in batch])).to(device)
            w = torch.from_numpy(np.stack([b['w'] for b in batch])).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model.pool(model.encode(x), slot, counts)
            loss = (F.binary_cross_entropy_with_logits(logits.float(), y, reduction='none') * w).sum() / w.sum()
            scaler.scale(loss / accum).backward()
            micro += 1
            n_images += x.shape[0]
            loss_sum += loss.item()
            loss_n += 1
            if micro % accum == 0:
                set_lr(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                step += 1
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            t_compute += t0 - t1
            if micro % 50 == 0:
                log(f'arm {arm} epoch {epoch} micro {micro} loss {loss.item():.4f} '
                    f'img/s {n_images / max(t_data + t_compute, 1e-9):.1f} data share {t_data / max(t_data + t_compute, 1e-9):.2f}')
            if time_up():
                stopped = True
                break
        epoch_seconds.append(round(time.time() - t_start, 1))
        receipt['elapsed_at_epoch_end'] = epoch_seconds
        if stopped:
            save_state('last', epoch, step)
            receipt.update(status='stopped_early', stopped_epoch=epoch, stopped_step=step)
            break
        if epoch + 1 >= CFG['eval_from_epoch']:
            z_val, s_val = run_eval(val_ids, Y_fit)
            rec_val, per_val = eval_record(val_ids, z_val, s_val)
            rec = {'val': rec_val, 'train_batch_loss_mean': loss_sum / max(loss_n, 1), 'train_probe': None}
            if probe_ids:
                z_tr, s_tr = run_eval(probe_ids, Y_fit)
                rec['train_probe'], _ = eval_record(probe_ids, z_tr, s_tr)
            receipt['epoch_eval'][str(epoch)] = rec
            tp = rec['train_probe'] or {}
            log(f'arm {arm} epoch {epoch} K{CFG["k_infer"]}: val AUC {rec_val["auc"]:.4f} loss {rec_val["loss"]:.4f} '
                f'({rec_val["seconds"]:.0f} s) | train probe AUC {tp.get("auc", float("nan")):.4f} '
                f'loss {tp.get("loss", float("nan")):.4f} ({tp.get("seconds", 0):.0f} s) | batch loss {rec["train_batch_loss_mean"]:.4f}')
            if rec_val['auc'] > best:
                best, best_epoch, best_per = rec_val['auc'], epoch, per_val
                torch.save({'model': model.state_dict(), 'epoch': epoch, 'cfg': CFG}, out / f'{stem}_best.pt')
                pd.DataFrame(sigmoid(z_val), columns=LABELS, index=pd.Index(val_ids, name='StudyInstanceUID')).to_csv(
                    out / f'{CFG["version"]}_oof_fold{fold}_{arm}_k{CFG["k_infer"]}.csv')
        write()

    elapsed = t_data + t_compute
    receipt.update(train_images=n_images, train_seconds=round(elapsed, 1),
                   train_images_per_s=round(n_images / max(elapsed, 1e-9), 1),
                   data_share=round(t_data / max(elapsed, 1e-9), 3), micro_steps=micro, optimizer_steps=step)
    if amp:
        receipt.update(peak_allocated_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                       peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1e9, 2))

    if best_epoch < 0:
        # No finite validation AUC (or stopped before the first evaluation): keep the last weights
        # and say so instead of leaving the receipt in 'running'.
        save_state('last', epoch, step)
        if receipt['status'] == 'running':
            receipt['status'] = 'no_valid_eval'
        receipt['total_seconds'] = round(time.time() - t_start, 1)
        write()
        log(json.dumps({k_: v for k_, v in receipt.items() if k_ != 'epoch_eval'}))
        return
    # Gold evaluation of the best checkpoint (also after an early stop, for the record only).
    model.load_state_dict(torch.load(out / f'{stem}_best.pt', map_location=device)['model'])
    z_gold, _ = run_eval(gold_ids, Y_soft)
    p_gold = sigmoid(z_gold)
    gyb = targets.loc[gold_ids, [f'{c}__gold' for c in LABELS]].to_numpy(np.float32)
    gm, gper = macro_auc(gyb, p_gold)
    pd.DataFrame(p_gold, columns=LABELS, index=pd.Index(gold_ids, name='StudyInstanceUID')).to_csv(
        out / f'{CFG["version"]}_gold_fold{fold}_{arm}_k{CFG["k_infer"]}.csv')
    receipt.update(status='done' if receipt['status'] == 'running' else 'stopped_early_evaluated',
                   best_epoch=best_epoch, val_macro_k16=best, val_per_label_k16=best_per,
                   gold_macro_k16=gm, gold_ci=bootstrap_ci(gyb, p_gold), gold_per_label=gper,
                   total_seconds=round(time.time() - t_start, 1))
    write()
    log(json.dumps({k_: v for k_, v in receipt.items() if k_ not in ('epoch_eval', 'val_per_label_k16', 'gold_per_label')}))


if __name__ == '__main__':
    main()

## Launch helpers and fold 0 (`fold0_arms`, `rerun_a`)

One process per arm, each on its own GPU. Each arm's time limit is its design limit, capped by
the time left before the global deadline (minus 15 minutes); it stops training, and an evaluation
already running may finish. A process is stopped and counts as incomplete if it is still running
5 minutes before the global deadline or `HANG_GRACE_S` after its own limit (hang guard). A stale
receipt from an earlier run in the same directory is removed at launch, and a non-zero exit code
marks the arm `failed`.

In [ ]:
HANG_GRACE_S = 3600   # normal overrun after the limit is one evaluation (about 17 min) plus gold

def remaining_s():
    return GLOBAL_DEADLINE_S - (time.time() - NOTEBOOK_T0)

def receipt_path(tag, fold, arm):
    return WORK / tag / f'v11_receipt_fold{fold}_{arm}.json'

def launch(arm, fold, gpu, time_limit_s):
    tag = f'fold{fold}_{arm}'
    receipt_path(tag, fold, arm).unlink(missing_ok=True)   # never read a receipt of an earlier run
    cfg = {**COMMON, **PATHS, **{k: v for k, v in ARMS[arm].items() if k not in ('gpu', 'expected_fold_s')},
           'arm': arm, 'fold': fold, 'time_limit_s': time_limit_s, 'out_dir': str(WORK / tag)}
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    logf = open(WORK / f'v11_log_{tag}.txt', 'w')
    proc = subprocess.Popen([sys.executable, '/kaggle/working/v11_train.py', json.dumps(cfg)],
                            env=env, stdout=logf, stderr=subprocess.STDOUT)
    print(f'launched {tag} on GPU {gpu} at {(time.time() - NOTEBOOK_T0) / 3600:.2f} h, '
          f'time limit {time_limit_s / 3600:.2f} h', flush=True)
    return {'tag': tag, 'arm': arm, 'fold': fold, 'proc': proc, 'logf': logf, 'killed': None,
            't0': time.time(), 'limit_s': time_limit_s}

def stop_if_overdue(job):
    if job['proc'].poll() is not None:
        return
    if remaining_s() < 5 * 60:
        job['killed'] = 'killed_at_deadline'
    elif time.time() - job['t0'] > job['limit_s'] + HANG_GRACE_S:
        job['killed'] = 'killed_hang_guard'
    else:
        return
    job['proc'].terminate()
    job['proc'].wait()
    print(f'{job["tag"]} stopped: {job["killed"]}', flush=True)

def collect(job):
    job['logf'].close()
    f = receipt_path(job['tag'], job['fold'], job['arm'])
    r = json.loads(f.read_text()) if f.is_file() else {'status': 'missing'}
    r['returncode'] = job['proc'].returncode
    if job['killed']:
        r['status_reported'], r['status'] = r.get('status'), job['killed']
    elif job['proc'].returncode != 0:
        r['status_reported'], r['status'] = r.get('status'), 'failed'
    print(f'{job["tag"]}: exit code {job["proc"].returncode}, status {r.get("status")}', flush=True)
    print(''.join(open(WORK / f'v11_log_{job["tag"]}.txt').readlines()[-6:]), flush=True)
    return r

def wait_all(jobs):
    out, pending = {}, list(jobs)
    while pending:
        for job in list(pending):
            stop_if_overdue(job)
            if job['proc'].poll() is not None:
                out[job['arm']] = collect(job)
                pending.remove(job)
        if pending:
            time.sleep(30)
    return out

stage = {}
if MODE in ('fold0_arms', 'rerun_a'):
    run_arms = ['A', 'B'] if MODE == 'fold0_arms' else ['A']
    jobs = [launch(a, FOLD, ARMS[a]['gpu'], min(ARMS[a]['time_limit_s'], remaining_s() - 15 * 60)) for a in run_arms]
    stage = wait_all(jobs)
    print(f'notebook elapsed {(time.time() - NOTEBOOK_T0) / 3600:.2f} h', flush=True)

## Gate R1 (design 4.2, revision 2.1)

Applied in this order: keep **complete** arms (receipt status `done`); keep **qualified** arms
(fold-0 val macro AUC at K_infer 16 >= 0.871); **pick one** arm by gold-58 macro AUC, arm A if
it is within 0.005 of the best; then the first matching rule decides on the picked arm's gold:

| # | Outcome | Action |
|---|---|---|
| 1 | No arm complete | Attempt 1: rerun arm A alone (`MODE = 'rerun_a'`); attempt 2: stop own-model work |
| 2 | Picked arm gold >= 0.915 | Train folds 1-4 with the picked arm |
| 3 | Picked arm gold in [0.905, 0.915) | Train folds 1-4 only if the remaining weekly quota covers the folds plus 20% (the user checks the quota) |
| 4 | Anything else | Stop own-model work |

The like-for-like table lists, per evaluation, validation and training-subset AUC and loss
against the training target (same checkpoint, K_infer 16, no augmentation).

In [ ]:
EPS = 1e-9   # numerical tolerance for the inclusive thresholds

def gate_r1(receipts, attempt, cfg=GATE_R1, arms=ARMS):
    def gold(a):
        g = receipts[a].get('gold_macro_k16')
        return float(g) if g is not None and np.isfinite(g) else float('nan')
    complete = [a for a in ('A', 'B') if receipts.get(a, {}).get('status') == 'done']
    qualified = [a for a in complete if (receipts[a].get('val_macro_k16') or -1.0) >= cfg['val_min'] - EPS
                 and np.isfinite(gold(a))]
    picked = None
    if qualified:
        best = max(gold(a) for a in qualified)
        picked = 'A' if 'A' in qualified and best - gold('A') <= cfg['tie'] + EPS else max(qualified, key=gold)
    d = {'attempt': attempt, 'complete': complete, 'qualified': qualified, 'picked_arm': picked,
         'arms': {a: {k: r.get(k) for k in ('status', 'best_epoch', 'val_macro_k16', 'gold_macro_k16', 'gold_ci',
                                            'total_seconds')} for a, r in receipts.items()}}
    if not complete:
        d.update(rule=1, action='rerun_arm_A' if attempt == 1 else 'stop_own_model')
    elif picked is not None and gold(picked) >= cfg['gold_go'] - EPS:
        d.update(rule=2, action='train_folds_1_4', arm=picked)
    elif picked is not None and gold(picked) >= cfg['gold_band_min'] - EPS:
        need_h = 2 * arms[picked]['expected_fold_s'] / 3600 * (1 + cfg['quota_margin'])
        d.update(rule=3, action='train_folds_1_4_if_quota', arm=picked, quota_needed_gpu_hours=round(need_h, 1))
    else:
        d.update(rule=4, action='stop_own_model')
    return d

def loss_table(receipts):
    rows = []
    for a, r in receipts.items():
        for e, rec in sorted(r.get('epoch_eval', {}).items(), key=lambda kv: int(kv[0])):
            tp = rec.get('train_probe') or {}
            rows.append({'arm': a, 'epoch': int(e), 'val_auc': rec['val']['auc'], 'val_loss': rec['val']['loss'],
                         'probe_auc': tp.get('auc'), 'probe_loss': tp.get('loss'),
                         'val_loss_soft': rec['val']['loss_soft'], 'probe_loss_soft': tp.get('loss_soft'),
                         'batch_loss': rec['train_batch_loss_mean']})
    return pd.DataFrame(rows)

if MODE in ('fold0_arms', 'rerun_a'):
    decision = gate_r1(stage, R1_ATTEMPT)
    table = loss_table(stage)
    decision['like_for_like'] = table.to_dict(orient='records')
    if len(table):
        print(table.round(4).to_string(index=False))
    print(json.dumps({k: v for k, v in decision.items() if k != 'like_for_like'}, indent=2, default=float))
    (WORK / f'v11_decision_r1_attempt{R1_ATTEMPT}.json').write_text(json.dumps(decision, indent=2, default=float))

## Fold queue (`folds`)

Trains the folds in `FOLDS_QUEUE` with `QUEUE_ARM` through a work queue over the two GPUs: a
fold starts as soon as a GPU is free and only if its expected duration, times 1.15, fits before
the global deadline (minus 15 minutes); otherwise it is skipped and listed. With arm B (about
7.6 h per fold) one session trains two folds, so run `FOLDS_QUEUE = [1, 2]` and then `[3, 4]`.

In [ ]:
queue_summary = {}
if MODE == 'folds':
    arm_cfg = ARMS[QUEUE_ARM]
    queue, running, skipped = list(FOLDS_QUEUE), {}, []
    while queue or running:
        for gpu in (0, 1):
            if gpu in running or not queue:
                continue
            left = remaining_s() - 15 * 60
            if left < 1.15 * arm_cfg['expected_fold_s']:
                skipped += queue
                print(f'not enough time for folds {queue}: {left / 3600:.2f} h left', flush=True)
                queue = []
                break
            running[gpu] = launch(QUEUE_ARM, queue.pop(0), gpu, min(arm_cfg['time_limit_s'], left))
        for gpu, job in list(running.items()):
            stop_if_overdue(job)
            if job['proc'].poll() is None:
                continue
            r = collect(job)
            queue_summary[job['tag']] = {k: r.get(k) for k in ('status', 'best_epoch', 'val_macro_k16', 'gold_macro_k16',
                                                                'gold_ci', 'total_seconds', 'val_per_label_k16')}
            del running[gpu]
        if running:
            time.sleep(30)
    queue_summary['_arm'] = QUEUE_ARM
    queue_summary['_skipped_folds'] = skipped
    queue_summary['_notebook_hours'] = round((time.time() - NOTEBOOK_T0) / 3600, 2)
    (WORK / 'v11_folds_summary.json').write_text(json.dumps(queue_summary, indent=2, default=float))
    print(json.dumps(queue_summary, indent=2, default=float))